In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================================
!pip -q install pandas numpy requests beautifulsoup4 pyarrow openpyxl thefuzz

In [ ]:
# ============================================================
# CARGA Y EXPLORACIÓN INICIAL DEL DATASET
# ============================================================
import pandas as pd
import numpy as np

FILE_PATH = "/content/interventions_l14-1.csv"

# Carga del CSV; on_bad_lines='warn' avisa si hay filas problemáticas
df = pd.read_csv(FILE_PATH, engine='python', on_bad_lines='warn')

print(type(df))
print(df.shape)
print(df.columns.tolist())
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: '/content/interventions_l14-1.csv'

In [ ]:
# ============================================================
# FILTRADO TEMPORAL: VENTANAS PRE Y POST ESTADO DE ALARMA
# ============================================================

import pandas as pd
import numpy as np

# df ya cargado con columnas: intervention, fecha, legislatura, namey, oradorx, role, oradorx_id

df = df.copy()
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce", dayfirst=True)

covid_start = pd.Timestamp("2020-03-14")
weeks_each_side = 10

pre_start  = covid_start - pd.Timedelta(weeks=weeks_each_side)
pre_end    = covid_start - pd.Timedelta(days=1)
post_start = covid_start
post_end   = covid_start + pd.Timedelta(weeks=weeks_each_side) - pd.Timedelta(days=1)

# Clasifica cada intervención según caiga en la ventana pre o post
df["period"] = np.where(
    (df["fecha"] >= pre_start) & (df["fecha"] <= pre_end), "pre",
    np.where((df["fecha"] >= post_start) & (df["fecha"] <= post_end), "post", None)
)

# Nos quedamos solo con observaciones dentro de las dos ventanas de análisis
df_w = df.dropna(subset=["period"]).copy()
print(df_w["period"].value_counts())
print("Rango:", df_w["fecha"].min(), "->", df_w["fecha"].max())

period
pre     1791
post    1608
Name: count, dtype: int64
Rango: 2020-01-04 00:00:00 -> 2020-05-20 00:00:00


In [ ]:
# ============================================================
# LIMPIEZA DEL TEXTO Y FILTRADO DE INTERVENCIONES BREVES
# ============================================================
import re

# Normaliza espacios y crea una versión limpia del texto
df_w["text"] = df_w["intervention"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df_w["n_words"] = df_w["text"].str.split().str.len()

# Elimina intervenciones demasiado cortas, normalmente poco informativas
df_w = df_w[df_w["n_words"] >= 30].copy()

print(df_w.shape)
df_w[["fecha","oradorx","role","period","n_words"]].head(5)

(1787, 10)


,fecha,oradorx,role,period,n_words
55,2020-01-04,BATET LAMAÑA,"PRESIDENTE, PRESIDENTA",pre,58
61,2020-01-04,ÁLVAREZ DE TOLEDO PERALTA-RAMOS,"DIPUTADO, DIPUTADA",pre,79
63,2020-01-04,BATET LAMAÑA,"PRESIDENTE, PRESIDENTA",pre,43
64,2020-01-04,PISARELLO PRADOS,SECRETARIO,pre,84
66,2020-01-04,SÁNCHEZ PÉREZ-CASTEJÓN,CANDIDATO A LA PRESIDENCIA DEL GOBIERNO,pre,3691


In [ ]:
df_w["intervention"] = df_w["intervention"].astype(str).str.strip()
df_w["n_words"] = df_w["intervention"].str.split().str.len()

In [ ]:
# ============================================================
# Diccionario local de unificación de partidos
# XIV Legislatura — construido desde datos públicos
# ============================================================

import pandas as pd
import unicodedata
from thefuzz import process, fuzz
from tqdm import tqdm

def normalizar(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFD", s.upper())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return " ".join(s.strip().split())  # elimina espacios dobles

# ============================================================
# Diccionario: apellidos normalizados --> (partido, grupo)
# Fuente: Congreso.es — Diputados XIV Legislatura
# ============================================================
DIPUTADOS_XIV = {
    # --- PSOE (Grupo Parlamentario Socialista) ---
    "SANCHEZ PEREZ-CASTEJON":           ("PSOE", "Grupo Parlamentario Socialista"),
    "SANCHEZ PEREZ":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "SANCHEZ PEREZCASTEJON":            ("PSOE", "Grupo Parlamentario Socialista"),
    "LASTRA FERNANDEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "BATET LAMANA":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "GUERRA LOPEZ":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "SIMANCAS SIMANCAS":                ("PSOE", "Grupo Parlamentario Socialista"),
    "RODRIGUEZ GOMEZ DE CELIS":         ("PSOE", "Grupo Parlamentario Socialista"),
    "GOMEZ DE CELIS":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "RODRIGUEZ DE CELIS":               ("PSOE", "Grupo Parlamentario Socialista"),
    "HERNANDO GARCIA":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "LLOP CUENCA":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA RODRIGUEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "RODRIGUEZ URIBES":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "ICETA I LLORENS":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "MARI KLOSE":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA MORIS":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "SAHUQUILLO GARCIA":                ("PSOE", "Grupo Parlamentario Socialista"),
    "PEDRENO MOLINA":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "SUMELZO JORDAN":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "CABEZON CASAS":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "HURTADO ZURERA":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "ROS MARTINEZ":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "SICILIA ALFEREZ":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "SILICIA ALFEREZ":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "MONTERO CUADRADO":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "ZARAGOZA ALONSO":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "RUIZ I CARBONELL":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "PEREA I CONILLAS":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "MEIJON COUSELO":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "CANCELA RODRIGUEZ":                ("PSOE", "Grupo Parlamentario Socialista"),
    "ALVAREZ PALLEIRO":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA GOMEZ":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "GONZALEZ RAMOS":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "VAQUERO PERIANEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA NIETO":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "MUNOZ VIDAL":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "FUENTES CURBELO":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "DIOUF DIOH":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "CARVALHO DANTAS":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "VALERIO CORDERO":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "ROBLES FERNANDEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "DELGADO ARCE":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "LEAL FERNANDEZ":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA GONZALEZ":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "CALVO GOMEZ":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "DARIAS SAN SEBASTIAN":             ("PSOE", "Grupo Parlamentario Socialista"),
    "CELAA DIEGUEZ":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "CARCEDO ROCES":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "CAMPO MORENO":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "CALVINO SANTAMARIA":               ("PSOE", "Grupo Parlamentario Socialista"),
    "ILLA ROCA":                        ("PSOE", "Grupo Parlamentario Socialista"),
    "MORANT RIPOLL":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "ALEGRIA CONTINENTE":               ("PSOE", "Grupo Parlamentario Socialista"),
    "MINONES CONDE":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "GABILONDO PUJOL":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "FERNANDEZ MARUGAN":                ("PSOE", "Grupo Parlamentario Socialista"),
    "JIMENEZ FERNANDEZ":                ("PSOE", "Grupo Parlamentario Socialista"),
    "SAEZ ALONSO MUNUMER":              ("PSOE", "Grupo Parlamentario Socialista"),
    "MARCOS DOMINGUEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "RENAU MARTINEZ":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "JOVER DIAZ":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "ELORZA GONZALEZ":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "VEGA ARIAS":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "BAENA PEDROSA":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "RAYA RODRIGUEZ":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "LORITE LORITE":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "SANCHEZ ESCOBAR":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "ZAPICO GONZALEZ":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "SOTO BURILLO":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA PUIG":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "SANTIAGO ROMERO":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "RUIZ ORTIZ":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "LOPEZ ZAMORA":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "SERRADA PARIENTE":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "PONS SAMPIETRO":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "ROMERO HERNANDEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "ALFARO GARCIA":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "ALONSO PEREZ":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "GUINART MORENO":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "GONZALEZ PEREZ":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA LOPEZ":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "CABALLERO GUTIERREZ":              ("PSOE", "Grupo Parlamentario Socialista"),
    "PADILLA RUIZ":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "PEDRAJA SAINZ":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "MIRALLES MARTIN":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "ALFONSO CENDON":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "JIMENEZ REVUELTA":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "SANCHEZ MARTIN":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "ANGUITA PEREZ":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "PIRIZ MAYA":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "ZAMBRANO GARCIA RAEZ":             ("PSOE", "Grupo Parlamentario Socialista"),
    "DIAZ PEREZ":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "FERNANDEZ CASTANON":               ("PSOE", "Grupo Parlamentario Socialista"),
    "FERNANDEZ BENITEZ":                ("PSOE", "Grupo Parlamentario Socialista"),
    "NARVAEZ BANDERA":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "CABALLERO MIGUEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "CALVO POYATO":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "RODRIGUEZ ALMEIDA":                ("PSOE", "Grupo Parlamentario Socialista"),
    "RUIZ NAVARRO":                     ("PSOE", "Grupo Parlamentario Socialista"),
    "ROMERO SANCHEZ":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "LIMA CID":                         ("PSOE", "Grupo Parlamentario Socialista"),
    "IBANEZ GARCIA":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "RAMOS ESTEBAN":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "DUQUE DUQUE":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "FERNANDEZ HERNANDEZ":              ("PSOE", "Grupo Parlamentario Socialista"),
    "CALVO LISTE":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA GURRUTXAGA":                ("PSOE", "Grupo Parlamentario Socialista"),
    "DIAZ MARIN":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "GARRIDO MARTINEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "CASANOVA PEIRO":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "HERNANDEZ GARCIA":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "GONZALEZ CABALLERO":               ("PSOE", "Grupo Parlamentario Socialista"),
    "GONZALEZ VAZQUEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "MARRA DOMINGUEZ":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "PEREZ RECUERDA":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA DIEZ":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "BRAVO BARCO":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "RUIZ LOPEZ":                       ("PSOE", "Grupo Parlamentario Socialista"),
    "MERINO MARTINEZ":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "FERNANDEZ PEREZ":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "FERNANDEZ RIOS":                   ("PSOE", "Grupo Parlamentario Socialista"),
    "GUERRA FERNANDEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "REDONDO CALVILLO":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "ROSELLO BOUSO":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "JIMENEZ LINUESA":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "SOLER MUR":                        ("PSOE", "Grupo Parlamentario Socialista"),
    "ARANDA VARGAS":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "RIBERA RODRIGUEZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "GARCIA CHAVARRIA":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "ZAMARRON MORENO":                  ("PSOE", "Grupo Parlamentario Socialista"),
    "SANCHEZ JODAR":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "CUATRECASAS ASUA":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "CANTERA DE CASTRO":                ("PSOE", "Grupo Parlamentario Socialista"),
    "BLANQUER ALCARAZ":                 ("PSOE", "Grupo Parlamentario Socialista"),
    "CRESPIN RUBIO":                    ("PSOE", "Grupo Parlamentario Socialista"),
    "CERDAN LEON":                      ("PSOE", "Grupo Parlamentario Socialista"),
    "GUTIERREZ SALINAS":                ("PSOE", "Grupo Parlamentario Socialista"),

    # --- PP (Grupo Parlamentario Popular) ---
    "CASADO BLANCO":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GARCIA EGEA":                      ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ALVAREZ DE TOLEDO PERALTA RAMOS":  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GAMARRA RUIZ CLAVIJO":             ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "PASTOR JULIAN":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ECHANIZ SALGADO":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "HERNANZ COSTA":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "SUAREZ ILLANA":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GONZALEZ TEROL":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MONEO DIEZ":                       ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "VELARDE GOMEZ":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "BERMUDEZ DE CASTRO":               ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MONTESINOS DE MIGUEL":             ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "JIMENEZ BECERRIL BARRIO":          ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "SEGADO MARTINEZ":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GARCIA PELAYO JURADO":             ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MORO ALMARAZ":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "RIOLOBOS REGADERA":                ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CARAZO HERMOSO":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "URIARTE TORREALDAY":               ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GONZALEZ LASO":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MARCOS ORTEGA":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ROJAS GARCIA":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "LOSADA FERNANDEZ":                 ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CANIZARES PACHECO":                ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "BELTRAN VILLALBA":                 ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "BETORET COLL":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "QUINTANILLA NAVARRO":              ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "BUENO CAMPANARIO":                 ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "NAVARRO LACOBA":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "PROHENS RIGO":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MALLADA DE CASTRO":                ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CASARES HONTANON":                 ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MANGLANO ALBACAR":                 ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MESTRE BAREA":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "HISPAN IGLESIAS DE USSEL":         ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MORALEJA GOMEZ":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CORTES CARBALLO":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ORIA LOPEZ":                       ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ELORRIAGA PISARIK":                ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ROBLES LOPEZ":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "TRIAS GIL":                        ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "BORRAS PABON":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "RAMIREZ DEL RIO":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MATARI SAEZ":                      ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "BUENO PINTO":                      ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "NEGRO KONRAD":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "NAVALPOTRO GOMEZ":                 ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "SANTAMARIA RUIZ":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MARCOS MORANO":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ALIAGA LOPEZ":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ANGULO ROMERO":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "DUQUE MORAN":                      ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "SAAVEDRA MUNOZ":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "RUEDA PERELLO":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CASTELLON RUBIO":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ACEVES GALINDO":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MOVELLAN LOMBILLA":                ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MARTINEZ FERRO":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "VAZQUEZ BLANCO":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MARISCAL ANAYA":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CLAVELL LOPEZ":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "DEL VALLE DE ISCAR":               ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "NEVADO DEL CAMPO":                 ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "BORREGO CORTES":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "HERRERO BONO":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ZURITA EXPOSITO":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "FERNANDEZ LOMANA GUTIERREZ":       ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ESTEBAN CALONJE":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MARI TORRES":                      ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "SAGRERAS BALLESTER":               ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "LAFUENTE MIR":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "BLANCO DE ANGULO":                 ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "LOPEZ CANO":                       ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "SANCHO INIGUEZ":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "OLANO VELA":                       ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CERQUEIRO GONZALEZ":               ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MARTINEZ DE TEJADA PEREZ":         ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ALVAREZ FANJUL":                   ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GUTIERREZ DIAZ DE OTAZU":          ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CRUZ GUZMAN GARCIA":               ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "MATEU ISTURIZ":                    ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ORTIZ GALVAN":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GONZALEZ GUINDA":                  ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GAGO BUGARIN":                     ("PP", "Grupo Parlamentario Popular en el Congreso"),

    # --- VOX (Grupo Parlamentario VOX) ---
    "ABASCAL CONDE":                    ("VOX", "Grupo Parlamentario VOX"),
    "ESPINOSA DE LOS MONTEROS":         ("VOX", "Grupo Parlamentario VOX"),
    "OLONA CHOCLAN":                    ("VOX", "Grupo Parlamentario VOX"),
    "GARRIGA VAZ":                      ("VOX", "Grupo Parlamentario VOX"),
    "GESTOSO DE MIGUEL":                ("VOX", "Grupo Parlamentario VOX"),
    "FIGAREDO ALVAREZ SALA":            ("VOX", "Grupo Parlamentario VOX"),
    "GIL LAZARO":                       ("VOX", "Grupo Parlamentario VOX"),
    "ORTEGA SMITH MOLINA":              ("VOX", "Grupo Parlamentario VOX"),
    "STEEGMANN OLMEDILLAS":             ("VOX", "Grupo Parlamentario VOX"),
    "CONTRERAS PELAEZ":                 ("VOX", "Grupo Parlamentario VOX"),
    "UTRILLA CANO":                     ("VOX", "Grupo Parlamentario VOX"),
    "TIRADO OCHOA":                     ("VOX", "Grupo Parlamentario VOX"),
    "REQUENA RUIZ":                     ("VOX", "Grupo Parlamentario VOX"),
    "VELASCO MORILLO":                  ("VOX", "Grupo Parlamentario VOX"),
    "CHAMORRO DELMO":                   ("VOX", "Grupo Parlamentario VOX"),
    "HOYO JULIA":                       ("VOX", "Grupo Parlamentario VOX"),
    "SALVA VERD":                       ("VOX", "Grupo Parlamentario VOX"),
    "SOTO SOTO":                        ("VOX", "Grupo Parlamentario VOX"),
    "PANIAGUA NUNEZ":                   ("VOX", "Grupo Parlamentario VOX"),
    "VILLANUEVA RUIZ":                  ("VOX", "Grupo Parlamentario VOX"),
    "PEREZ ABELLAS":                    ("VOX", "Grupo Parlamentario VOX"),
    "DE LUNA TOBARRA":                  ("VOX", "Grupo Parlamentario VOX"),
    "LOPEZ MOYA":                       ("VOX", "Grupo Parlamentario VOX"),
    "MANSO OLIVAR":                     ("VOX", "Grupo Parlamentario VOX"),
    "GONZALEZ COELLO DE PORTUGAL":      ("VOX", "Grupo Parlamentario VOX"),
    "CONESA ALCARAZ":                   ("VOX", "Grupo Parlamentario VOX"),
    "MEDEL PEREZ":                      ("VOX", "Grupo Parlamentario VOX"),
    "VILCHES RUIZ":                     ("VOX", "Grupo Parlamentario VOX"),
    "AIZCORBE TORRA":                   ("VOX", "Grupo Parlamentario VOX"),
    "BUIL GARCIA":                      ("VOX", "Grupo Parlamentario VOX"),
    "TAIBO MONELOS":                    ("VOX", "Grupo Parlamentario VOX"),
    "PEDRAZA SAINZ":                    ("VOX", "Grupo Parlamentario VOX"),
    "SUAREZ LAMATA":                    ("VOX", "Grupo Parlamentario VOX"),
    "TARNO BLANCO":                     ("VOX", "Grupo Parlamentario VOX"),
    "ALCARAZ MARTOS":                   ("VOX", "Grupo Parlamentario VOX"),
    "BOADELLA ESTEVE":                  ("VOX", "Grupo Parlamentario VOX"),
    "MUNOZ DALDA":                      ("VOX", "Grupo Parlamentario VOX"),
    "MOLINA GALLARDO":                  ("VOX", "Grupo Parlamentario VOX"),
    "FRANCO CARMONA":                   ("VOX", "Grupo Parlamentario VOX"),
    "DE SIMON CABALLERO":               ("VOX", "Grupo Parlamentario VOX"),
    "DE MEER MENDEZ":                   ("VOX", "Grupo Parlamentario VOX"),
    "SANCHEZ DEL REAL":                 ("VOX", "Grupo Parlamentario VOX"),
    "ROSETY FERNANDEZ DE CASTRO":       ("VOX", "Grupo Parlamentario VOX"),
    "MENDEZ MONASTERIO":                ("VOX", "Grupo Parlamentario VOX"),
    "SANCHEZ GARCIA":                   ("VOX", "Grupo Parlamentario VOX"),
    "LOPEZ MARAVER":                    ("VOX", "Grupo Parlamentario VOX"),
    "DE LAS HERAS FERNANDEZ":           ("VOX", "Grupo Parlamentario VOX"),
    "MARQUEZ GUERRERO":                 ("VOX", "Grupo Parlamentario VOX"),
    "JEREZ JUAN":                       ("VOX", "Grupo Parlamentario VOX"),
    "MARISCAL ZABALA":                  ("VOX", "Grupo Parlamentario VOX"),
    "POSTIGO QUINTANA":                 ("VOX", "Grupo Parlamentario VOX"),
    "GUAITA ESTERUELAS":                ("VOX", "Grupo Parlamentario VOX"),
    "SEVA RUIZ":                        ("VOX", "Grupo Parlamentario VOX"),
    "CALLEJAS CANO":                    ("VOX", "Grupo Parlamentario VOX"),
    "DELGADO RAMOS":                    ("VOX", "Grupo Parlamentario VOX"),
    "GAMAZO MICO":                      ("VOX", "Grupo Parlamentario VOX"),
    "MARQUEZ ROMERO":                   ("VOX", "Grupo Parlamentario VOX"),
    "ASARTA CUEVAS":                    ("VOX", "Grupo Parlamentario VOX"),
    "RUIZ SOLAS":                       ("VOX", "Grupo Parlamentario VOX"),
    "TOSCANO DE BALBIN":                ("VOX", "Grupo Parlamentario VOX"),
    "REQUEJO NOVOA":                    ("VOX", "Grupo Parlamentario VOX"),

    # --- Ciudadanos (Cs) ---
    "ARRIMADAS GARCIA":                 ("Cs", "Grupo Parlamentario Ciudadanos"),
    "BAL FRANCES":                      ("Cs", "Grupo Parlamentario Ciudadanos"),
    "DE QUINTO ROMERO":                 ("Cs", "Grupo Parlamentario Ciudadanos"),
    "ESPEJO SAAVEDRA CONESA":           ("Cs", "Grupo Parlamentario Ciudadanos"),
    "MARTINEZ GRANADOS":                ("Cs", "Grupo Parlamentario Ciudadanos"),
    "CAMBRONERO PIQUERAS":              ("Cs", "Grupo Parlamentario Ciudadanos"),
    "GUTIERREZ VIVAS":                  ("Cs", "Grupo Parlamentario Ciudadanos"),
    "RIVERA RODRIGUEZ":                 ("Cs", "Grupo Parlamentario Ciudadanos"),
    "CORTES GOMEZ":                     ("Cs", "Grupo Parlamentario Ciudadanos"),
    "ALMODEBAR BARCELO":                ("Cs", "Grupo Parlamentario Ciudadanos"),
    "PENA CAMARERO":                    ("Cs", "Grupo Parlamentario Ciudadanos"),
    "REYES RIVERA":                     ("Cs", "Grupo Parlamentario Ciudadanos"),
    "ANTON CACHO":                      ("Cs", "Grupo Parlamentario Ciudadanos"),
    "LOPEZ BAS VALERO":                 ("Cs", "Grupo Parlamentario Ciudadanos"),
    "GIMENEZ GIMENEZ":                  ("Cs", "Grupo Parlamentario Ciudadanos"),
    "DIAZ GOMEZ":                       ("Cs", "Grupo Parlamentario Ciudadanos"),
    "PICO GARCES":                      ("Cs", "Grupo Parlamentario Ciudadanos"),
    "MARTIN LLAGUNO":                   ("Cs", "Grupo Parlamentario Ciudadanos"),

    # --- Unidas Podemos / Confederal ---
    "MONTERO GIL":                      ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "PITA CARDENES":                    ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "MENA ARCA":                        ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "ELIZO SERRANO":                    ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "IGLESIAS TURRION":                 ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "ECHENIQUE ROBBA":                  ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "GARZON ESPINOSA":                  ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "GOMEZ REINO VARELA":               ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "ASENS LLODRA":                     ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "MAESTRO MOLINER":                  ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "MAYORAL PERALES":                  ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "VIDAL SAEZ":                       ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "LOPEZ DE URALDE GARMENDIA":        ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "ALBIACH SATORRES":                 ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "NUET PUJALS":                      ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "JARA MORENO":                      ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "GUIJARRO GARCIA":                  ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "SANCHEZ SERNA":                    ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "PISARELLO PRADOS":                 ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "TIZON VAZQUEZ":                    ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "BELARRA URTEAGA":                  ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "SUBIRATS HUMET":                   ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "ARAUJO MORALES":                   ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "DOMINGUEZ GONZALEZ":               ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "DALMAU DE MATA":                   ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "MIGUEZ GARCIA":                    ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "GUIJARRO CEBALLOS":                ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "CARCEDO GARCIA":                   ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "HONRUBIA HURTADO":                 ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "FERNANDEZ GARCIA":                 ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "MARI RIPA":                        ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "BUSTAMANTE MARTIN":                ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "VALLINA DE LA NOVAL":              ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "GARRIDO GUTIERREZ":                ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),

    # --- ERC (Republicano) ---
    "RUFIAN ROMERO":                    ("ERC", "Grupo Parlamentario Republicano"),
    "ERITJA CIURO":                     ("ERC", "Grupo Parlamentario Republicano"),
    "PAGES I MASSO":                    ("ERC", "Grupo Parlamentario Republicano"),
    "PUJOL I FARRE":                    ("ERC", "Grupo Parlamentario Republicano"),
    "MARGALL SASTRE":                   ("ERC", "Grupo Parlamentario Republicano"),
    "SEGURA JUST":                      ("ERC", "Grupo Parlamentario Republicano"),
    "ROSIQUE I SALTOR":                 ("ERC", "Grupo Parlamentario Republicano"),
    "CAPDEVILA I ESTEVE":               ("ERC", "Grupo Parlamentario Republicano"),
    "SALVADOR I DUCH":                  ("ERC", "Grupo Parlamentario Republicano"),
    "BASSA COLL":                       ("ERC", "Grupo Parlamentario Republicano"),
    "ALVAREZ I GARCIA":                 ("ERC", "Grupo Parlamentario Republicano"),
    "GRANOLLERS CUNILLERA":             ("ERC", "Grupo Parlamentario Republicano"),

    # --- JxCat / PDeCAT (Plural) ---
    "BORRAS CASTANYER":                 ("JxCat", "Grupo Parlamentario Plural"),
    "NOGUERAS I CAMERO":                ("JxCat", "Grupo Parlamentario Plural"),
    "CANADELL SALVIA":                  ("JxCat", "Grupo Parlamentario Plural"),
    "TELECHEA I LOZANO":                ("JxCat", "Grupo Parlamentario Plural"),
    "VALLUGERA BALANA":                 ("JxCat", "Grupo Parlamentario Plural"),
    "MIQUEL I VALENTI":                 ("JxCat", "Grupo Parlamentario Plural"),
    "PUJOL I BONELL":                   ("JxCat", "Grupo Parlamentario Plural"),
    "ALONSO CUEVILLAS I SAYROL":        ("JxCat", "Grupo Parlamentario Plural"),
    "ILLAMOLA DAUSA":                   ("JxCat", "Grupo Parlamentario Plural"),
    "BEL ACCENSI":                      ("PDeCAT", "Grupo Parlamentario Plural"),

    # --- PNV (Vasco) ---
    "ESTEBAN BRAVO":                    ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),
    "LEGARDA URIARTE":                  ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),
    "AGIRRETXEA URRESTI":               ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),
    "BARANDIARAN BENITO":               ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),
    "GOROSPE ELEZCANO":                 ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),
    "ELEZCANO GOROSPE":                 ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),
    "SAGASTIZABAL UNZETABARRENETXEA":   ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),
    "IPINAZAR MIRANDA":                 ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),

    # --- EH Bildu ---
    "POZUETA FERNANDEZ":                ("EH Bildu", "Grupo Parlamentario Euskal Herria Bildu"),
    "AIZPURUA ARZALLUS":                ("EH Bildu", "Grupo Parlamentario Euskal Herria Bildu"),
    "MATUTE GARCIA DE JALON":           ("EH Bildu", "Grupo Parlamentario Euskal Herria Bildu"),
    "OSKAR MATUTE":                     ("EH Bildu", "Grupo Parlamentario Euskal Herria Bildu"),
    "BANOS RUIZ":                       ("EH Bildu", "Grupo Parlamentario Euskal Herria Bildu"),
    "INARRITU GARCIA":                  ("EH Bildu", "Grupo Parlamentario Euskal Herria Bildu"),
    "RUIZ DE PINEDO UNDIANO":           ("EH Bildu", "Grupo Parlamentario Euskal Herria Bildu"),

    # --- Otros Grupos (Plural / Mixto) ---
    "ERREJON GALVAN":                   ("Más País", "Grupo Parlamentario Más País-Equo"),
    "SABANES NADAL":                    ("Más País", "Grupo Parlamentario Más País-Equo"),
    "VEHI CANTENYS":                    ("CUP", "Grupo Parlamentario Plural"),
    "BOTRAN PAHISSA":                   ("CUP", "Grupo Parlamentario Plural"),
    "REGO CANDAMIL":                    ("BNG", "Grupo Parlamentario Plural"),
    "PONTON MONDELO":                   ("BNG", "Grupo Parlamentario Plural"),
    "ORAMAS GONZALEZ MORO":             ("CC", "Grupo Parlamentario Plural"),
    "QUEVEDO ITURBE":                   ("NC/CC", "Grupo Parlamentario Plural"),
    "BALDOVI RODA":                     ("Compromís", "Grupo Parlamentario Plural"),
    "MAZON RAMOS":                      ("PRC", "Grupo Parlamentario Regionalista"),
    "GUITARTE GIMENO":                  ("Teruel Existe", "Grupo Parlamentario Mixto"),
    "GARCIA ADANERO":                   ("UPN", "Grupo Parlamentario Mixto"),
    "SAYAS LOPEZ":                      ("UPN", "Grupo Parlamentario Mixto"),
    "MARTINEZ OBLANCA":                 ("Foro Asturias", "Grupo Parlamentario Mixto"),
    "RODRIGUEZ RODRIGUEZ":              ("Mixto", "Grupo Parlamentario Mixto"),

    # --- Gobierno / Mesa ---
    "PRESIDENTE DEL GOBIERNO":          ("Gobierno-PSOE", "Gobierno"),
    "VICEPRESIDENTA PRIMERA":           ("Gobierno-PSOE", "Gobierno"),
    "VICEPRESIDENTE SEGUNDO":           ("Gobierno-UP", "Gobierno"),
    "VICEPRESIDENTA SEGUNDA":           ("Gobierno-UP", "Gobierno"),
    "VICEPRESIDENTA TERCERA":           ("Gobierno-PSOE", "Gobierno"),
    "PRESIDENTA":                       ("Mesa Congreso", "Mesa del Congreso"),
    "GRANDE MARLASKA":                  ("Gobierno-PSOE", "Gobierno"),
    "MINISTRA DE TRABAJO":              ("Gobierno-UP", "Gobierno"),
    "CASTELLS OLIVAN":                  ("Gobierno-UP", "Gobierno"),
}
# Ajustes y sobreescrituras manuales para casos conflictivos detectados
DIPUTADOS_XIV.update({
    "CARVALHO DANTAS": ("ERC", "Grupo Parlamentario Republicano"),
    "SANTIAGO ROMERO": ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "DIAZ PEREZ": ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "PIRIZ MAYA": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GARCIA DIEZ": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "FERNANDEZ CASTANON": ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "JOVER DIAZ": ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "LORITE LORITE": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "CASARES HONTANON": ("PSOE", "Grupo Parlamentario Socialista"),
    "CANIZARES PACHECO": ("VOX", "Grupo Parlamentario VOX"),
})
# Ajustes y sobreescrituras manuales para casos conflictivos detectados
DIPUTADOS_XIV.update({
    "BERMÚDEZ DE CASTRO FERNÁNDEZ": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "DE OLANO VELA": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ESPAÑA REINA": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ESPINOSA DE LOS MONTEROS DE SIMÓN": ("VOX", "Grupo Parlamentario VOX"),
    "FAGÚNDEZ CAMPO": ("PSOE", "Grupo Parlamentario Socialista"),
    "FERNÁNDEZ-ROCA SUÁREZ": ("VOX", "Grupo Parlamentario VOX"),
    "GARCÉS SANAGUSTÍN": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "GARRIGA VAZ DE CONCICAO": ("VOX", "Grupo Parlamentario VOX"),
    "GOROSPE LEZCANO": ("PNV", "Grupo Parlamentario Vasco (EAJ-PNV)"),
    "GRANDE-MARLASKA GÓMEZ": ("Gobierno-PSOE", "Gobierno"),
    "GRANDEMARLASKA GÓMEZ": ("Gobierno-PSOE", "Gobierno"),
    "GÓMEZ HERNÁNDEZ": ("PSOE", "Grupo Parlamentario Socialista"),
    "LAMUÀ ESTAÑOL": ("PSOE", "Grupo Parlamentario Socialista"),
    "LÓPEZ DOMÍNGUEZ": ("PSOE", "Grupo Parlamentario Socialista"),
    "LÓPEZ ÁLVAREZ": ("PSOE", "Grupo Parlamentario Socialista"),
    "MINISTRA DE ASUNTOS EXTERIORES": ("Gobierno-PSOE", "Gobierno"),
    "MINISTRA DE INDUSTRIA": ("Gobierno-PSOE", "Gobierno"),
    "MINISTRO DE AGRICULTURA": ("Gobierno-PSOE", "Gobierno"),
    "MINISTRO DE INCLUSIÓN": ("Gobierno-PSOE", "Gobierno"),
    "MINISTRO DE TRANSPORTES": ("Gobierno-PSOE", "Gobierno"),
    "MONTESINOS AGUAYO": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "NAVARRO LÓPEZ": ("PSOE", "Grupo Parlamentario Socialista"),
    "RODRÍGUEZ HERRER": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "ROMERO VILCHES": ("VOX", "Grupo Parlamentario VOX"),
    "SARRIÀ MORELL": ("PSOE", "Grupo Parlamentario Socialista"),
    "URIARTE BENGOECHEA": ("PP", "Grupo Parlamentario Popular en el Congreso"),
    "VICEPRESIDENTA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA": ("Gobierno-PSOE", "Gobierno"),
    "VICEPRESIDENTA PRIMERA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA": ("Gobierno-PSOE", "Gobierno"),
    "VICEPRESIDENTE SEGUNDO DEL GOBIERNO Y MINISTRO DE DERECHO SOCIALES Y AGENDA": ("Gobierno-UP", "Gobierno"),
    "VICEPRESIDENTE SEGUNDO DEL GOBIERNO Y MINISTRO DE DERECHOS SOCIALES Y AGENDA": ("Gobierno-UP", "Gobierno"),
})
# Ajustes y sobreescrituras manuales para casos conflictivos detectados
DIPUTADOS_XIV.update({
    "GARCIA PUIG": ("Unidas Podemos", "Grupo Parlamentario Confederal de Unidas Podemos"),
    "ANGUITA PEREZ": ("PSOE", "Grupo Parlamentario Socialista"),
})
# Se normalizan todas las claves para facilitar el matching posterior
DIPUTADOS_NORM = {normalizar(k): v for k, v in DIPUTADOS_XIV.items()}
CLAVES = list(DIPUTADOS_NORM.keys())

print(f"Entradas en el diccionario: {len(DIPUTADOS_NORM)}")

Entradas en el diccionario: 427


In [ ]:
# ============================================================
# Clasificador con fuzzy match
# ============================================================
from thefuzz import process, fuzz

UMBRAL = 88

def clasificar(orador_raw: str) -> dict:
    key = normalizar(orador_raw)

   # 1. Coincidencia exacta
    if key in DIPUTADOS_NORM:
        partido, grupo = DIPUTADOS_NORM[key]
        return {"partido": partido, "grupo": grupo, "match": orador_raw, "score": 100}

   # 2. Coincidencia aproximada si no hay match exacto
    resultado = process.extractOne(key, CLAVES, scorer=fuzz.token_sort_ratio)
    if resultado and resultado[1] >= UMBRAL:
        mejor_key = resultado[0]
        partido, grupo = DIPUTADOS_NORM[mejor_key]
        return {"partido": partido, "grupo": grupo, "match": mejor_key, "score": resultado[1]}

    return {"partido": "REVISAR", "grupo": "", "match": "", "score": 0}

# ============================================================
# Aplicación del clasificador al DataFrame
# ============================================================
df = pd.read_csv("/content/drive/MyDrive/interventions_l14-1.csv")
COLUMNA = "oradorx"

# Se clasifica una sola vez por orador único y luego se reutiliza el resultado
oradores_unicos = df[COLUMNA].dropna().unique()
cache = {o: clasificar(o) for o in tqdm(oradores_unicos, desc="Clasificando")}

df["partido"]  = df[COLUMNA].map(lambda x: cache.get(x, {}).get("partido", ""))
df["grupo"]    = df[COLUMNA].map(lambda x: cache.get(x, {}).get("grupo", ""))
df["_match"]   = df[COLUMNA].map(lambda x: cache.get(x, {}).get("match", ""))
df["_score"]   = df[COLUMNA].map(lambda x: cache.get(x, {}).get("score", 0))

# ============================================================
# Diagnóstico y exportación
# ============================================================
pendientes = df[df["partido"] == "REVISAR"][[COLUMNA]].drop_duplicates()
print(f"Pendientes de revisión: {len(pendientes)}")
print(pendientes.head(109).to_string())

# Casos con fuzzy match aceptado pero con score relativamente bajo
dudosos = df[(df["_score"] > 0) & (df["_score"] < 92)].drop_duplicates(COLUMNA)[[COLUMNA,"partido","_match","_score"]]
print(f"\nMatches por revisar (score < 92): {len(dudosos)}")
print(dudosos.to_string())

df.to_csv("intervenciones_con_partido.csv", index=False)
print("\nGuardado.")

Clasificando: 100%|██████████| 607/607 [00:00<00:00, 1327.66it/s]


Pendientes de revisión: 81
                                                                                                                              oradorx
937                                                                                                                        JOSÉ VÉLEZ
1394                                                                                                                 ORTEGA DOMÍNGUEZ
1396                                                                                                                     AZORÍN SALAR
1398                                                                                                                    SENDEROS ORAÁ
1408                                                                                                                     LÓPEZ SOMOZA
1410                                                                                                                   ARRIBAS MAROTO
3907                               

In [ ]:
# ============================================================
# RECALCULAR CLASIFICACIÓN, REVISAR PENDIENTES Y GUARDAR
# ============================================================

# Recalcula la clasificación por orador único y actualiza las columnas finales
for orador in df[COLUMNA].dropna().unique():
    cache[orador] = clasificar(orador)

df["partido"] = df[COLUMNA].map(lambda x: cache.get(x, {}).get("partido", ""))
df["grupo"]   = df[COLUMNA].map(lambda x: cache.get(x, {}).get("grupo", ""))
df["_match"]  = df[COLUMNA].map(lambda x: cache.get(x, {}).get("match", ""))
df["_score"]  = df[COLUMNA].map(lambda x: cache.get(x, {}).get("score", 0))

aun_pendientes = df[df["partido"] == "REVISAR"][[COLUMNA]].drop_duplicates()
print(f"Aún sin clasificar: {len(aun_pendientes)}")
if len(aun_pendientes):
    print(aun_pendientes.to_string())

print("\nIntervenciones por partido:")
print(df["partido"].value_counts().to_string())

df.to_csv("intervenciones_con_partido.csv", index=False)
print("\nGuardado.")

Aún sin clasificar: 81
                                                                                                                              oradorx
937                                                                                                                        JOSÉ VÉLEZ
1394                                                                                                                 ORTEGA DOMÍNGUEZ
1396                                                                                                                     AZORÍN SALAR
1398                                                                                                                    SENDEROS ORAÁ
1408                                                                                                                     LÓPEZ SOMOZA
1410                                                                                                                   ARRIBAS MAROTO
3907                                   

In [ ]:
# ============================================================
# DIAGNÓSTICO: ORADORES ÚNICOS Y REVISIÓN DE CASOS PSOE
# ============================================================

print("Oradores únicos por partido:")
print(df.groupby("partido")[COLUMNA].nunique().sort_values(ascending=False).to_string())

print("\nTop oradores PSOE por número de intervenciones:")
print(df[df["partido"] == "PSOE"][COLUMNA].value_counts().head(50).to_string())

# Revisa asignaciones de PSOE que no hayan quedado marcadas como exactas
print("\nPSOE no exactos:")
print(
    df[(df["partido"] == "PSOE") & (df["_match"] != "exacto_diccionario")][[COLUMNA, "_match", "_score"]]
    .drop_duplicates()
    .sort_values(COLUMNA)
    .to_string(index=False)
)

Oradores únicos por partido:
partido
PSOE              156
PP                107
REVISAR            81
VOX                78
Unidas Podemos     45
PNV                23
Cs                 23
ERC                17
JxCat              17
Gobierno-PSOE      16
EH Bildu           12
CUP                 6
Gobierno-UP         5
CC                  3
Compromís           3
Más País            2
BNG                 2
PDeCAT              2
UPN                 2
NC/CC               2
Foro Asturias       1
Mesa Congreso       1
Mixto               1
PRC                 1
Teruel Existe       1

Top oradores PSOE por número de intervenciones:
oradorx
BATET LAMAÑA                18416
RODRÍGUEZ GÓMEZ DE CELIS     6591
SÁNCHEZ PÉREZ-CASTEJÓN        635
MONTERO CUADRADO              386
CALVIÑO SANTAMARÍA            366
MUÑOZ VIDAL                   239
RIBERA RODRÍGUEZ              222
DARIAS SAN SEBASTIÁN          106
CALVO GÓMEZ                    88
CAMPO MORENO                   86
ROBLES FERNÁNDEZ

In [ ]:
# ============================================================
# Reasignar Gobierno al partido y excluir Mesa Congreso
# ============================================================

GOBIERNO_MAP = {
    "Gobierno-PSOE": "PSOE",
    "Gobierno-UP":   "Unidas Podemos",
}
# Reasigna intervenciones del Gobierno al partido correspondiente
df["partido_limpio"] = df["partido"].replace(GOBIERNO_MAP)

# Excluye la Mesa del Congreso del análisis
df = df[df["partido"] != "Mesa Congreso"].copy()

print(df["partido_limpio"].value_counts().to_string())


# ============================================================
# Parsear fecha y crear variable pre/post
# ============================================================
import pandas as pd

COLUMNA_FECHA = "fecha"
ESTADO_ALARMA = pd.Timestamp("2020-03-14")

df["fecha_dt"] = pd.to_datetime(df[COLUMNA_FECHA], format="%d/%m/%Y", errors="coerce")

n_nulos = df["fecha_dt"].isna().sum()
print(f"Filas con fecha no parseada: {n_nulos}")
if n_nulos > 0:
    print(df[df["fecha_dt"].isna()][COLUMNA_FECHA].unique()[:10])

# Clasifica cada intervención como pre o post 14/03/2020
df["periodo"] = df["fecha_dt"].apply(
    lambda x: "pre" if pd.notna(x) and x < ESTADO_ALARMA else "post"
)

print(f"\nPre  (antes del 14 mar 2020): {(df['periodo']=='pre').sum()} intervenciones")
print(f"Post (desde el 14 mar 2020):  {(df['periodo']=='post').sum()} intervenciones")
print(f"Rango de fechas: {df['fecha_dt'].min().date()} → {df['fecha_dt'].max().date()}")


# ============================================================
# Tabla de volumen pre vs post por partido
# ============================================================

PARTIDOS_PRINCIPALES = [
    "PSOE", "PP", "VOX", "Cs", "Unidas Podemos",
    "PNV", "ERC", "JxCat", "BNG", "EH Bildu",
    "CUP", "Compromís", "Más País", "PRC", "CC",
    "Teruel Existe", "UPN", "Foro Asturias", "NC/CC", "PDeCAT"
]

df_main = df[df["partido_limpio"].isin(PARTIDOS_PRINCIPALES)].copy()

tabla = (
    df_main.groupby(["partido_limpio", "periodo"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"pre": "pre_alarma", "post": "post_alarma"})
)

tabla["total"] = tabla["pre_alarma"] + tabla["post_alarma"]
tabla["pct_pre"] = (tabla["pre_alarma"] / tabla["total"] * 100).round(1)
tabla["pct_post"] = (tabla["post_alarma"] / tabla["total"] * 100).round(1)

# Cambio relativo del volumen post respecto al pre
tabla["cambio_pct"] = (
    (tabla["post_alarma"] - tabla["pre_alarma"]) / tabla["pre_alarma"].replace(0, 1) * 100
).round(1)

tabla = tabla.sort_values("total", ascending=False)

print("\nVolumen de intervenciones pre vs post estado de alarma:")
print(tabla.to_string())


# ============================================================
# Normalizar por semanas
# ============================================================

semanas_pre  = (ESTADO_ALARMA - df_main["fecha_dt"].min()).days / 7
semanas_post = (df_main["fecha_dt"].max() - ESTADO_ALARMA).days / 7

print(f"\nSemanas pre:  {semanas_pre:.1f}")
print(f"Semanas post: {semanas_post:.1f}")

tabla["interv_x_semana_pre"] = (tabla["pre_alarma"] / semanas_pre).round(1)
tabla["interv_x_semana_post"] = (tabla["post_alarma"] / semanas_post).round(1)

print("\nIntervenciones por semana (normalizado):")
print(
    tabla[["interv_x_semana_pre", "interv_x_semana_post", "cambio_pct"]]
    .sort_values("cambio_pct")
    .to_string()
)


# ============================================================
# Guardar tabla resumen
# ============================================================
tabla.to_csv("volumen_pre_post_alarma.csv")
df.to_csv("intervenciones_con_partido.csv", index=False)
print("\nGuardado.")

partido_limpio
PSOE              30302
PP                 3352
VOX                2633
Unidas Podemos     2306
Cs                 1416
PNV                1069
ERC                1034
REVISAR             782
EH Bildu            741
JxCat               655
BNG                 582
Foro Asturias       447
CUP                 406
Compromís           372
UPN                 332
Más País            268
PDeCAT              252
PRC                 241
Teruel Existe       212
CC                  199
NC/CC                92
Mixto                18
Filas con fecha no parseada: 0

Pre  (antes del 14 mar 2020): 1845 intervenciones
Post (desde el 14 mar 2020):  45866 intervenciones
Rango de fechas: 2019-12-03 → 2023-08-16

Volumen de intervenciones pre vs post estado de alarma:
periodo         post_alarma  pre_alarma  total  pct_pre  pct_post  cambio_pct
partido_limpio                                                               
PSOE                  29085        1217  30302      4.0      96.0     

In [ ]:
# ============================================================
# Unir partido a df_w, reasignar Gobierno y excluir Mesa Congreso
# ============================================================

# Clasifica los oradores presentes en df_w
oradores_w = df_w["oradorx"].dropna().unique()
cache_w = {o: clasificar(o) for o in oradores_w}

df_w["partido"] = df_w["oradorx"].map(lambda x: cache_w.get(x, {}).get("partido", "REVISAR"))

GOBIERNO_MAP = {
    "Gobierno-PSOE": "PSOE",
    "Gobierno-UP":   "Unidas Podemos",
}

# Reasigna intervenciones del Gobierno al partido correspondiente
df_w["partido"] = df_w["partido"].replace(GOBIERNO_MAP)

# Excluye Mesa Congreso del análisis
df_w = df_w[df_w["partido"] != "Mesa Congreso"].copy()

print(df_w["partido"].value_counts().to_string())
print(f"\nPendientes: {(df_w['partido']=='REVISAR').sum()}")

partido
PSOE              765
PP                187
VOX               122
Unidas Podemos    120
ERC                70
Cs                 68
PNV                62
EH Bildu           59
JxCat              44
CUP                40
BNG                37
Más País           35
Compromís          31
Foro Asturias      30
PRC                24
CC                 22
UPN                21
PDeCAT             19
Teruel Existe      18
NC/CC              12
Mixto               1

Pendientes: 0


In [ ]:
# ============================================================
# Reclasificar df_w corrigiendo Mesa Congreso de forma explícita
# ============================================================


# Clasifica los oradores presentes en df_w
oradores_w = df_w["oradorx"].dropna().unique()
cache_w = {o: clasificar(o) for o in oradores_w}

df_w["partido"] = df_w["oradorx"].map(lambda x: cache_w.get(x, {}).get("partido", "REVISAR"))

# Reasigna intervenciones del Gobierno al partido correspondiente
GOBIERNO_MAP = {
    "Gobierno-PSOE": "PSOE",
    "Gobierno-UP":   "Unidas Podemos",
}
df_w["partido"] = df_w["partido"].replace(GOBIERNO_MAP)

# Fuerza como Mesa Congreso ciertos nombres institucionales
# que estaban contaminando principalmente el PSOE
mask_mesa = df_w["oradorx"].astype(str).str.contains(
    r"BATET|GOMEZ DE CELIS|GÓMEZ DE CELIS|RODRIGUEZ DE CELIS|RODRÍGUEZ DE CELIS",
    case=False,
    na=False
)

df_w.loc[mask_mesa, "partido"] = "Mesa Congreso"

# Excluye Mesa Congreso del análisis
df_w = df_w[df_w["partido"] != "Mesa Congreso"].copy()

print(df_w["partido"].value_counts().to_string())
print(f"\nPendientes: {(df_w['partido']=='REVISAR').sum()}")

partido
PSOE              449
PP                187
VOX               122
Unidas Podemos    120
ERC                70
Cs                 68
PNV                62
EH Bildu           59
JxCat              44
CUP                40
BNG                37
Más País           35
Compromís          31
Foro Asturias      30
PRC                24
CC                 22
UPN                21
PDeCAT             19
Teruel Existe      18
NC/CC              12
Mixto               1

Pendientes: 0


In [ ]:
# ============================================================
# GUARDAR DATASET FINAL CON PARTIDOS UNIFICADOS
# ============================================================
df_w.to_csv("intervenciones_10semanas_con_partido_final.csv", index=False)

In [ ]:
# ============================================================
# DIAGNÓSTICO: DISTRIBUCIÓN FINAL Y CASOS BATET/CELIS
# ============================================================
print(df_w["partido"].value_counts().to_string())

# Comprueba cómo han quedado clasificados los nombres asociados a Mesa Congreso
print(
    df_w[df_w["oradorx"].astype(str).str.contains("BATET|CELIS", case=False, na=False)]
    [["oradorx", "partido"]]
    .drop_duplicates()
    .to_string(index=False)
)

partido
PSOE              449
PP                187
VOX               122
Unidas Podemos    120
ERC                70
Cs                 68
PNV                62
EH Bildu           59
JxCat              44
CUP                40
BNG                37
Más País           35
Compromís          31
Foro Asturias      30
PRC                24
CC                 22
UPN                21
PDeCAT             19
Teruel Existe      18
NC/CC              12
Mixto               1
Empty DataFrame
Columns: [oradorx, partido]
Index: []


In [ ]:
df_w["oradorx"] = df_w["oradorx"].replace({
    "SÁNCHEZ PÉREZ-CASTEJON": "SÁNCHEZ PÉREZ-CASTEJÓN",
    "SANCHEZ PÉREZ-CASTEJÓN": "SÁNCHEZ PÉREZ-CASTEJÓN",
    "CALVINO SANTAMARÍA": "CALVIÑO SANTAMARÍA",
})

In [ ]:
# ============================================================
# Corrección de oradorx/namey/role: cargos --> nombres reales
# ============================================================
NOMBRES_REALES = {
    "MINISTRO DE TRANSPORTES":                                                          ("ÁBALOS ÍÑIGO",              "MINISTRO DE TRANSPORTES"),
    "MINISTRA DE INDUSTRIA":                                                            ("MAROTO ILLERA",             "MINISTRA DE INDUSTRIA"),
    "MINISTRO DE AGRICULTURA":                                                          ("PLANAS PUCHADES",           "MINISTRO DE AGRICULTURA"),
    "MINISTRO DE INCLUSIÓN":                                                            ("ESCRIVÁ BELMONTE",          "MINISTRO DE INCLUSIÓN"),
    "MINISTRA DE ASUNTOS EXTERIORES":                                                   ("GONZÁLEZ LAYA",             "MINISTRA DE ASUNTOS EXTERIORES"),
    "PRESIDENTE DEL GOBIERNO":                                                          ("SÁNCHEZ PÉREZ-CASTEJÓN",    "PRESIDENTE DEL GOBIERNO"),
    "VICEPRESIDENTA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA":                         ("CALVO POYATO",              "VICEPRESIDENTA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA"),
    "VICEPRESIDENTA PRIMERA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA":                 ("CALVO POYATO",              "VICEPRESIDENTA PRIMERA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA"),
    "VICEPRESIDENTE SEGUNDO DEL GOBIERNO Y MINISTRO DE DERECHO SOCIALES Y AGENDA ":    ("IGLESIAS TURRIÓN",          "MINISTRO DE DERECHOS SOCIALES"),
    "VICEPRESIDENTE SEGUNDO DEL GOBIERNO Y MINISTRO DE DERECHOS SOCIALES Y AGENDA ":   ("IGLESIAS TURRIÓN",          "MINISTRO DE DERECHOS SOCIALES"),
}

# Sustituye cargos institucionales por el nombre real del orador
for cargo, (nombre, rol) in NOMBRES_REALES.items():
    mask = df_w["oradorx"] == cargo
    df_w.loc[mask, "oradorx"] = nombre
    df_w.loc[mask, "namey"]   = nombre
    df_w.loc[mask, "role"]    = rol

# Verificación de los cambios aplicados
print("=== Correcciones aplicadas ===")
nombres = [v[0] for v in NOMBRES_REALES.values()]
print(df_w[df_w["oradorx"].isin(nombres)][["oradorx","role","partido","period"]]
      .drop_duplicates()
      .sort_values("oradorx")
      .to_string())
print(f"\nTotal filas corregidas: {df_w['oradorx'].isin(nombres).sum()}")

# Caso adicional: corrige el role de Iglesias si ya tenía el nombre bien
# pero seguía apareciendo como diputado
df.loc[(df["oradorx"] == "IGLESIAS TURRIÓN") &
       (df["role"] == "DIPUTADO, DIPUTADA"), "role"] = "MINISTRO DE DERECHOS SOCIALES"

=== Correcciones aplicadas ===
                     oradorx                                                              role         partido period
1208            CALVO POYATO          VICEPRESIDENTA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA            PSOE    pre
1204            CALVO POYATO  VICEPRESIDENTA PRIMERA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA            PSOE    pre
2973            CALVO POYATO  VICEPRESIDENTA PRIMERA DEL GOBIERNO Y MINISTRA DE LA PRESIDENCIA            PSOE   post
427         ESCRIVÁ BELMONTE                                             MINISTRO DE INCLUSIÓN            PSOE    pre
1316           GONZÁLEZ LAYA                                    MINISTRA DE ASUNTOS EXTERIORES            PSOE    pre
808         IGLESIAS TURRIÓN                                     MINISTRO DE DERECHOS SOCIALES  Unidas Podemos    pre
117         IGLESIAS TURRIÓN                                                DIPUTADO, DIPUTADA  Unidas Podemos    pre
2254        IGLESIAS TURR

In [ ]:
from google.colab import files
df_w.to_csv("intervenciones_10semanas_con_partido_final.csv", index=False)
files.download("intervenciones_10semanas_con_partido_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>